*If you see ![](http://training.databricks.com/databricks_guide/ImportNotebookIcon3.png) at the top-left or top-right, click on the link to import this notebook in order to run it.*

# On-Time Flight Performance with GraphFrames for Apache Spark
This notebook provides an analysis of On-Time Flight Performance and Departure Delays data using GraphFrames for Apache Spark.  This notebook has been updated to use Apache Spark 2.0 and GraphFrames 0.3.
* Original blog post: [On-Time Flight Performance with GraphFrames with Apache Spark Blog Post](https://databricks.com/blog/2016/03/16/on-time-flight-performance-with-graphframes-for-apache-spark.html)
* Original Notebook: [On-Time Flight Performance with GraphFrames with Apache Spark Notebook](http://cdn2.hubspot.net/hubfs/438089/notebooks/Samples/Miscellaneous/On-Time_Flight_Performance.html)


Source Data: 
* [OpenFlights: Airport, airline and route data](http://openflights.org/data.html)
* [United States Department of Transportation: Bureau of Transportation Statistics (TranStats)](http://www.transtats.bts.gov/DL_SelectFields.asp?Table_ID=236&DB_Short_Name=On-Time)
 * Note, the data used here was extracted from the US DOT:BTS between 1/1/2014 and 3/31/2014*

References:
* [GraphFrames User Guide](http://graphframes.github.io/user-guide.html)
* [GraphFrames: DataFrame-based Graphs (GitHub)](https://github.com/graphframes/graphframes)
* [D3 Airports Example](http://mbostock.github.io/d3/talk/20111116/airports.html)
* [MLlib and Machine Learning: Binary Classification](https://docs.databricks.com/spark/latest/mllib/binary-classification-mllib-pipelines.html)

### Preparation
Extract the Airports and Departure Delays information from S3 / DBFS

### Kiểm tra dữ liệu đầu vào
Cell này liệt kê các file dữ liệu trong thư mục `flight-data` để xác nhận notebook đang đọc đúng nguồn dữ liệu cục bộ.

In [161]:
from pathlib import Path

base_dir = Path(r"c:\Users\as\Downloads\Graph Mining - On Time Flight\flight-data")
print(sorted(path.name for path in base_dir.iterdir()))

['airport-codes-na.txt', 'departuredelays.csv']


### Khởi tạo Spark và nạp dữ liệu
Cell này tạo `SparkSession` local, đọc hai file dữ liệu vào DataFrame và đăng ký các temp view để những truy vấn SQL phía sau dùng lại.

In [162]:
from pathlib import Path
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("OnTimeFlightPerformance").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

base_dir = Path(r"c:\Users\as\Downloads\Graph Mining - On Time Flight\flight-data")
tripdelaysFilePath = str(base_dir / "departuredelays.csv")
airportsnaFilePath = str(base_dir / "airport-codes-na.txt")

# Obtain airports dataset
airportsna = spark.read.csv(airportsnaFilePath, header=True, inferSchema=True, sep='\t')
airportsna.createOrReplaceTempView("airports_na")

# Obtain departure Delays data
departureDelays = spark.read.csv(tripdelaysFilePath, header=True, inferSchema=True)
departureDelays.createOrReplaceTempView("departureDelays")
departureDelays.cache()

# Available IATA codes from the departuredelays sample dataset
tripIATA = spark.sql("select distinct iata from (select distinct origin as iata from departureDelays union all select distinct destination as iata from departureDelays) a")
tripIATA.createOrReplaceTempView("tripIATA")

# Only include airports with at least one trip from the departureDelays dataset
airports = spark.sql("select f.IATA, f.City, f.State, f.Country from airports_na f join tripIATA t on t.IATA = f.IATA")
airports.createOrReplaceTempView("airports")
airports = airports.cache()

### Đếm số bản ghi chuyến bay gốc
Con số trả về ở cell kế tiếp là tổng số dòng trong file `departuredelays.csv`, tức tổng số bản ghi chuyến bay có trong bộ dữ liệu.

In [163]:
departureDelays.count()

1391578

### Ghép dữ liệu chuyến bay với thông tin sân bay
Cell này tạo bảng `departureDelays_geo` bằng cách gắn thời gian, độ trễ, khoảng cách và thông tin origin/destination vào từng chuyến bay.

In [164]:
# Build `departureDelays_geo` DataFrame
#  Obtain key attributes such as Date of flight, delays, distance, and airport information (Origin, Destination)
departureDelays_geo = spark.sql("""
select cast(f.date as int) as tripid,
       to_timestamp(concat('2014-', substr(lpad(cast(f.date as string), 8, '0'), 1, 2), '-', substr(lpad(cast(f.date as string), 8, '0'), 3, 2), ' ', substr(lpad(cast(f.date as string), 8, '0'), 5, 2), ':', substr(lpad(cast(f.date as string), 8, '0'), 7, 2), ':00')) as localdate,
       cast(f.delay as int) as delay,
       cast(f.distance as int) as distance,
       f.origin as src,
       f.destination as dst,
       o.city as city_src,
       d.city as city_dst,
       o.state as state_src,
       d.state as state_dst
from departuredelays f
join airports o on o.iata = f.origin
join airports d on d.iata = f.destination
""")

# Create Temporary View and cache
departureDelays_geo.createOrReplaceTempView("departureDelays_geo")
departureDelays_geo.cache()

# Count
departureDelays_geo.count()

1361141

### Kiểm tra dữ liệu đã chuẩn hóa
Cell này hiển thị vài dòng đầu của `departureDelays_geo` để kiểm tra các cột như thời gian bay, độ trễ và sân bay đi/đến.

In [165]:
# Review the top 10 rows of the `departureDelays_geo` DataFrame
departureDelays_geo.show(10)

+-------+-------------------+-----+--------+---+---+---------+--------+---------+---------+
| tripid|          localdate|delay|distance|src|dst| city_src|city_dst|state_src|state_dst|
+-------+-------------------+-----+--------+---+---+---------+--------+---------+---------+
|1011245|2014-01-01 12:45:00|    6|     602|ABE|ATL|Allentown| Atlanta|       PA|       GA|
|1020600|2014-01-02 06:00:00|   -8|     369|ABE|DTW|Allentown| Detroit|       PA|       MI|
|1021245|2014-01-02 12:45:00|   -2|     602|ABE|ATL|Allentown| Atlanta|       PA|       GA|
|1020605|2014-01-02 06:05:00|   -4|     602|ABE|ATL|Allentown| Atlanta|       PA|       GA|
|1031245|2014-01-03 12:45:00|   -4|     602|ABE|ATL|Allentown| Atlanta|       PA|       GA|
|1030605|2014-01-03 06:05:00|    0|     602|ABE|ATL|Allentown| Atlanta|       PA|       GA|
|1041243|2014-01-04 12:43:00|   10|     602|ABE|ATL|Allentown| Atlanta|       PA|       GA|
|1040605|2014-01-04 06:05:00|   28|     602|ABE|ATL|Allentown| Atlanta|       PA

### Xem thêm một mẫu dữ liệu đã làm sạch
Cell này hiển thị thêm một phần của `departureDelays_geo` để kiểm tra dữ liệu sau khi chuẩn hóa.

### Xem mẫu dữ liệu dạng bảng
Cell này in một mẫu của `departureDelays_geo`; mỗi dòng là một chuyến bay đã được ghép đủ thông tin.

In [137]:
# Using `display` to view the data
departureDelays_geo.show(10, truncate=False)

+-------+-------------------+-----+--------+---+---+---------+--------+---------+---------+
|tripid |localdate          |delay|distance|src|dst|city_src |city_dst|state_src|state_dst|
+-------+-------------------+-----+--------+---+---+---------+--------+---------+---------+
|1011245|2014-01-01 12:45:00|6    |602     |ABE|ATL|Allentown|Atlanta |PA       |GA       |
|1020600|2014-01-02 06:00:00|-8   |369     |ABE|DTW|Allentown|Detroit |PA       |MI       |
|1021245|2014-01-02 12:45:00|-2   |602     |ABE|ATL|Allentown|Atlanta |PA       |GA       |
|1020605|2014-01-02 06:05:00|-4   |602     |ABE|ATL|Allentown|Atlanta |PA       |GA       |
|1031245|2014-01-03 12:45:00|-4   |602     |ABE|ATL|Allentown|Atlanta |PA       |GA       |
|1030605|2014-01-03 06:05:00|0    |602     |ABE|ATL|Allentown|Atlanta |PA       |GA       |
|1041243|2014-01-04 12:43:00|10   |602     |ABE|ATL|Allentown|Atlanta |PA       |GA       |
|1040605|2014-01-04 06:05:00|28   |602     |ABE|ATL|Allentown|Atlanta |PA       

## Building the Graph
Now that we've imported our data, we're going to need to build our graph. To do so we're going to do two things: we are going to build the structure of the vertices (or nodes) and we're going to build the structure of the edges. What's awesome about GraphFrames is that this process is incredibly simple. 
* Rename IATA airport code to **id** in the Vertices Table
* Start and End airports to **src** and **dst** for the Edges Table (flights)

These are required naming conventions for vertices and edges in GraphFrames as of the time of this writing (Feb. 2016).

**WARNING:** If the graphframes package, required in the cell below, is not installed, follow the instructions [here](http://cdn2.hubspot.net/hubfs/438089/notebooks/help/Setup_graphframes_package.html).

### Chuẩn bị lớp GraphFrame cục bộ
Cell này định nghĩa một bản GraphFrame thay thế để notebook chạy trong môi trường local, không phụ thuộc vào gói GraphFrames của Databricks.

In [166]:
from dataclasses import dataclass
import builtins
import re

import networkx as nx
from pyspark.sql import functions as F


def _struct_from_row(alias, columns):
    return F.struct(*[F.col(f"{alias}.{column}").alias(column) for column in columns]).alias(alias)


@dataclass
class PageRankResult:
    vertices: object
    edges: object


class GraphFrame:
    def __init__(self, vertices, edges):
        self.vertices = vertices
        self.edges = edges

    @property
    def inDegrees(self):
        return self.edges.groupBy("dst").count().withColumnRenamed("dst", "id").withColumnRenamed("count", "inDegree")

    @property
    def outDegrees(self):
        return self.edges.groupBy("src").count().withColumnRenamed("src", "id").withColumnRenamed("count", "outDegree")

    @property
    def degrees(self):
        return self.inDegrees.select("id", F.col("inDegree").alias("degree")).unionByName(
            self.outDegrees.select("id", F.col("outDegree").alias("degree"))
        ).groupBy("id").sum("degree").withColumnRenamed("sum(degree)", "degree")

    def pageRank(self, resetProbability=0.15, maxIter=5):
        edge_rows = self.edges.groupBy("src", "dst").count().collect()
        graph = nx.DiGraph()
        for edge_row in edge_rows:
            graph.add_edge(edge_row["src"], edge_row["dst"], weight=edge_row["count"])
        for vertex_row in self.vertices.select("id").collect():
            graph.add_node(vertex_row["id"])

        if graph.number_of_nodes() == 0:
            pagerank_map = {}
        else:
            iteration_limit = builtins.max(100, maxIter * 20)
            try:
                pagerank_map = nx.pagerank(
                    graph,
                    alpha=1.0 - resetProbability,
                    max_iter=iteration_limit,
                    weight="weight",
                )
            except nx.PowerIterationFailedConvergence:
                pagerank_map = nx.pagerank(graph, alpha=1.0 - resetProbability, weight="weight", tol=1e-6)

        pagerank_rows = [(vertex_id, float(pagerank_map.get(vertex_id, 0.0))) for vertex_id in graph.nodes()]
        pagerank_frame = self.vertices.sparkSession.createDataFrame(pagerank_rows, ["id", "pagerank"])
        ranked_vertices = self.vertices.join(pagerank_frame, on="id", how="left").fillna({"pagerank": 0.0})
        return PageRankResult(vertices=ranked_vertices, edges=self.edges)

    def find(self, pattern):
        normalized_pattern = pattern.replace(" ", "")
        if normalized_pattern != "(a)-[ab]->(b);(b)-[bc]->(c)":
            raise NotImplementedError("This local notebook only supports the motif used in the notebook.")

        vertices_a = self.vertices.alias("a")
        vertices_b = self.vertices.alias("b")
        vertices_c = self.vertices.alias("c")
        edges_ab = self.edges.alias("ab")
        edges_bc = self.edges.alias("bc")

        return (
            edges_ab.join(edges_bc, F.col("ab.dst") == F.col("bc.src"))
            .join(vertices_a, F.col("ab.src") == F.col("a.id"))
            .join(vertices_b, F.col("ab.dst") == F.col("b.id"))
            .join(vertices_c, F.col("bc.dst") == F.col("c.id"))
            .select(
                _struct_from_row("a", self.vertices.columns),
                _struct_from_row("ab", self.edges.columns),
                _struct_from_row("b", self.vertices.columns),
                _struct_from_row("bc", self.edges.columns),
                _struct_from_row("c", self.vertices.columns),
            )
        )

    def bfs(self, fromExpr, toExpr, maxPathLength=1):
        source_match = re.search(r"id\s*=\s*'([^']+)'", fromExpr)
        target_match = re.search(r"id\s*=\s*'([^']+)'", toExpr)
        if not source_match or not target_match:
            raise ValueError("Only id equality expressions are supported in this local notebook.")

        source_id = source_match.group(1)
        target_id = target_match.group(1)
        source_vertices = self.vertices.filter(F.col("id") == source_id).alias("v0")
        target_vertices = self.vertices.filter(F.col("id") == target_id).alias("v1")

        if maxPathLength <= 1:
            matching_edges = self.edges.filter((F.col("src") == source_id) & (F.col("dst") == target_id)).alias("e0")
            return matching_edges.crossJoin(source_vertices).crossJoin(target_vertices).select(
                _struct_from_row("v0", self.vertices.columns),
                _struct_from_row("e0", self.edges.columns),
                _struct_from_row("v1", self.vertices.columns),
            )

        first_hop = self.edges.alias("e0")
        second_hop = self.edges.alias("e1")
        middle_vertices = self.vertices.alias("v1")
        target_vertices = self.vertices.alias("v2")
        matching_paths = (
            first_hop.join(second_hop, F.col("e0.dst") == F.col("e1.src"))
            .filter((F.col("e0.src") == source_id) & (F.col("e1.dst") == target_id))
            .join(source_vertices, F.lit(True))
            .join(middle_vertices, F.col("e0.dst") == F.col("v1.id"))
            .join(target_vertices, F.lit(True))
        )

        return matching_paths.select(
            _struct_from_row("v0", self.vertices.columns),
            _struct_from_row("e0", self.edges.columns),
            _struct_from_row("v1", self.vertices.columns),
            _struct_from_row("e1", self.edges.columns),
            _struct_from_row("v2", self.vertices.columns),
        )

### Tạo danh sách đỉnh và cạnh của đồ thị
Cell này đổi bảng sân bay thành vertices (`id`) và bảng chuyến bay thành edges (`src`, `dst`) để dựng đồ thị.

In [167]:
# Note, the local notebook uses a lightweight GraphFrame implementation defined above.
from pyspark.sql.functions import *

# Create Vertices (airports) and Edges (flights)
tripVertices = airports.withColumnRenamed("IATA", "id").distinct()
tripEdges = departureDelays_geo.select("tripid", "delay", "src", "dst", "city_dst", "state_dst")

# Cache Vertices and Edges
tripEdges.cache()
tripVertices = tripVertices.cache()

### Xem dữ liệu vertices và edges
Hai cell tiếp theo cho thấy danh sách sân bay (vertices) và chuyến bay (edges) đã được chuẩn hóa cho đồ thị.

In [168]:
# Vertices
#   The vertices of our graph are the airports
tripVertices.show(10, truncate=False)

+---+-------------------+-----+-------+
|id |City               |State|Country|
+---+-------------------+-----+-------+
|INL|International Falls|MN   |USA    |
|MSY|New Orleans        |LA   |USA    |
|GEG|Spokane            |WA   |USA    |
|BUR|Burbank            |CA   |USA    |
|SNA|Orange County      |CA   |USA    |
|GRB|Green Bay          |WI   |USA    |
|GTF|Great Falls        |MT   |USA    |
|IDA|Idaho Falls        |ID   |USA    |
|GRR|Grand Rapids       |MI   |USA    |
|JLN|Joplin             |MO   |USA    |
+---+-------------------+-----+-------+
only showing top 10 rows


### Danh sách vertices
Cell này hiển thị các sân bay trong đồ thị; mỗi dòng là một vertex, không phải một số đo.

In [169]:
# Edges
#  The edges of our graph are the flights between airports
tripEdges.show(10, truncate=False)

+-------+-----+---+---+--------+---------+
|tripid |delay|src|dst|city_dst|state_dst|
+-------+-----+---+---+--------+---------+
|1011245|6    |ABE|ATL|Atlanta |GA       |
|1020600|-8   |ABE|DTW|Detroit |MI       |
|1021245|-2   |ABE|ATL|Atlanta |GA       |
|1020605|-4   |ABE|ATL|Atlanta |GA       |
|1031245|-4   |ABE|ATL|Atlanta |GA       |
|1030605|0    |ABE|ATL|Atlanta |GA       |
|1041243|10   |ABE|ATL|Atlanta |GA       |
|1040605|28   |ABE|ATL|Atlanta |GA       |
|1051245|88   |ABE|ATL|Atlanta |GA       |
|1050605|9    |ABE|ATL|Atlanta |GA       |
+-------+-----+---+---+--------+---------+
only showing top 10 rows


### Danh sách edges
Cell này hiển thị các chuyến bay giữa sân bay đi và sân bay đến; đây là dữ liệu cạnh của đồ thị.

In [170]:
# Build `tripGraph` GraphFrame
#  This GraphFrame builds up on the vertices and edges based on our trips (flights)
tripGraph = GraphFrame(tripVertices, tripEdges)

# Build `tripGraphPrime` GraphFrame
#   This graphframe contains a smaller subset of data to make it easier to display motifs and subgraphs (below)
tripEdgesPrime = departureDelays_geo.select("tripid", "delay", "src", "dst")
tripGraphPrime = GraphFrame(tripVertices, tripEdgesPrime)

### Khởi tạo đồ thị GraphFrame
Cell này tạo `tripGraph` và `tripGraphPrime`, là hai đồ thị dùng cho các truy vấn phía sau.

## Simple Queries
Let's start with a set of simple graph queries to understand flight performance and departure delays

#### Determine the number of airports and trips

In [171]:
print("Airports: %d" % tripGraph.vertices.count())
print("Trips: %d" % tripGraph.edges.count())

Airports: 279
Trips: 1361141


### Đếm số sân bay và số chuyến bay
Hai con số in ra lần lượt là tổng số vertex trong đồ thị và tổng số edge/chuyến bay.

#### Determining the longest delay in this dataset

### Tìm độ trễ lớn nhất
Cell này lấy giá trị delay lớn nhất trong toàn bộ cạnh của đồ thị; con số đó là số phút trễ lớn nhất quan sát được.

In [172]:
tripGraph.edges.groupBy().max("delay").show()

+----------+
|max(delay)|
+----------+
|      1642|
+----------+



### Xem bảng độ trễ lớn nhất
Cell này hiển thị kết quả tổng hợp `max(delay)`; giá trị đó chính là số phút trễ cao nhất trong dữ liệu.

In [146]:
# Finding the longest Delay
longestDelay = tripGraph.edges.groupBy().max("delay")
longestDelay.show()

+----------+
|max(delay)|
+----------+
|      1642|
+----------+



#### Determining the number of delayed vs. on-time / early flights

### Đếm chuyến đúng giờ/đi sớm và chuyến trễ
Hai con số in ra lần lượt là số chuyến có delay <= 0 và số chuyến có delay > 0.

In [173]:
# Determining number of on-time / early flights vs. delayed flights
print("On-time / Early Flights: %d" % tripGraph.edges.filter("delay <= 0").count())
print("Delayed Flights: %d" % tripGraph.edges.filter("delay > 0").count())

On-time / Early Flights: 780469
Delayed Flights: 580672


#### What flights departing SEA are most likely to have significant delays
Note, delay can be <= 0 meaning the flight left on time or early

### Các chuyến bay từ SEA có độ trễ cao
Cell này lọc các chuyến bay đi từ Seattle và tính độ trễ trung bình theo từng điểm đến.

In [174]:
tripGraph.edges\
  .filter("src = 'SEA' and delay > 0")\
  .groupBy("src", "dst")\
  .avg("delay")\
  .sort(desc("avg(delay)"))\
  .show(5)
  

+---+---+------------------+
|src|dst|        avg(delay)|
+---+---+------------------+
|SEA|PHL|55.666666666666664|
|SEA|COS| 43.53846153846154|
|SEA|FAT| 43.03846153846154|
|SEA|LGB| 39.39705882352941|
|SEA|IAD|37.733333333333334|
+---+---+------------------+
only showing top 5 rows


### Trung bình độ trễ từ SEA theo từng điểm đến
Bảng in ra cho biết từng tuyến SEA -> dst có độ trễ trung bình bao nhiêu phút; giá trị lớn hơn nghĩa là tuyến đó hay bị trễ hơn.

In [149]:
tripGraph.edges.filter("src = 'SEA' and delay > 0").groupBy("src", "dst").avg("delay").sort(desc("avg(delay)")).show(10, truncate=False)

+---+---+------------------+
|src|dst|avg(delay)        |
+---+---+------------------+
|SEA|PHL|55.666666666666664|
|SEA|COS|43.53846153846154 |
|SEA|FAT|43.03846153846154 |
|SEA|LGB|39.39705882352941 |
|SEA|IAD|37.733333333333334|
|SEA|MIA|37.325581395348834|
|SEA|SFO|36.50210378681627 |
|SEA|SBA|36.48275862068966 |
|SEA|JFK|35.03125          |
|SEA|ORD|33.60335195530726 |
+---+---+------------------+
only showing top 10 rows


#### What destinations tend to have delays

### Các chuyến bay có delay dương
Cell này lọc các chuyến bay bị trễ để xem chúng tập trung ở những điểm đến nào.

In [175]:
# After viewing tripDelays, use Plot Options to set `state_dst` as a Key.
tripDelays = tripGraph.edges.filter("delay > 0")
tripDelays.show(10, truncate=False)

+-------+-----+---+---+--------+---------+
|tripid |delay|src|dst|city_dst|state_dst|
+-------+-----+---+---+--------+---------+
|1011245|6    |ABE|ATL|Atlanta |GA       |
|1041243|10   |ABE|ATL|Atlanta |GA       |
|1040605|28   |ABE|ATL|Atlanta |GA       |
|1051245|88   |ABE|ATL|Atlanta |GA       |
|1050605|9    |ABE|ATL|Atlanta |GA       |
|1061725|69   |ABE|ATL|Atlanta |GA       |
|1081230|33   |ABE|DTW|Detroit |MI       |
|1080625|1    |ABE|ATL|Atlanta |GA       |
|1080607|5    |ABE|ORD|Chicago |IL       |
|1081219|54   |ABE|ORD|Chicago |IL       |
+-------+-----+---+---+--------+---------+
only showing top 10 rows


### Danh sách các chuyến bay bị trễ
Cell này hiển thị các cạnh có delay > 0; mỗi dòng là một chuyến bay trễ với thông tin nguồn, đích và độ trễ.

#### What destinations tend to have significant delays departing from SEA

### Các chuyến từ SEA trễ trên 100 phút
Cell này lọc các chuyến SEA có delay rất cao để xem các điểm đến tệ nhất.

In [176]:
# States with the longest cumulative delays (with individual delays > 100 minutes) (origin: Seattle)
tripGraph.edges.filter("src = 'SEA' and delay > 100").show(10, truncate=False)

+-------+-----+---+---+--------+---------+
|tripid |delay|src|dst|city_dst|state_dst|
+-------+-----+---+---+--------+---------+
|1021425|298  |SEA|ORD|Chicago |IL       |
|1030600|103  |SEA|ORD|Chicago |IL       |
|1060830|116  |SEA|DFW|Dallas  |TX       |
|1081205|135  |SEA|ORD|Chicago |IL       |
|1112205|101  |SEA|MIA|Miami   |FL       |
|1161205|582  |SEA|ORD|Chicago |IL       |
|1180830|132  |SEA|DFW|Dallas  |TX       |
|1241205|174  |SEA|ORD|Chicago |IL       |
|1262205|130  |SEA|MIA|Miami   |FL       |
|1281350|115  |SEA|DFW|Dallas  |TX       |
+-------+-----+---+---+--------+---------+
only showing top 10 rows


## Vertex Degrees
* `inDegrees`: Incoming connections to the airport
* `outDegrees`: Outgoing connections from the airport 
* `degrees`: Total connections to and from the airport

Reviewing the various properties of the property graph to understand the incoming and outgoing connections between airports.

### Phân tích degree của sân bay
Ba cell tiếp theo cho biết tổng số kết nối, số kết nối vào và số kết nối ra của từng sân bay; đây là các thống kê cấu trúc của đồ thị.

In [152]:
# Degrees
#  The number of degrees - the number of incoming and outgoing connections - for various airports within this sample dataset
tripGraph.degrees.sort(desc("degree")).limit(20).show(20, truncate=False)

+---+------+
|id |degree|
+---+------+
|ATL|179774|
|DFW|133966|
|ORD|125405|
|LAX|106853|
|DEN|103699|
|IAH|85685 |
|PHX|79672 |
|SFO|77635 |
|LAS|66101 |
|CLT|56103 |
|EWR|54407 |
|MCO|54300 |
|LGA|50927 |
|SLC|50780 |
|BOS|49936 |
|DTW|46705 |
|MSP|46235 |
|SEA|45816 |
|JFK|43661 |
|BWI|42526 |
+---+------+



### Degree tổng
Cell này xếp hạng sân bay theo tổng số kết nối vào + ra; số càng lớn nghĩa là sân bay càng kết nối nhiều.

In [153]:
# inDegrees
#  The number of degrees - the number of incoming connections - for various airports within this sample dataset
tripGraph.inDegrees.sort(desc("inDegree")).limit(20).show(20, truncate=False)

+---+--------+
|id |inDegree|
+---+--------+
|ATL|89633   |
|DFW|65767   |
|ORD|61654   |
|LAX|53184   |
|DEN|50738   |
|IAH|42512   |
|PHX|39619   |
|SFO|38641   |
|LAS|32994   |
|CLT|28044   |
|EWR|27201   |
|MCO|27071   |
|LGA|25469   |
|SLC|25169   |
|BOS|24973   |
|DTW|23297   |
|SEA|22906   |
|MSP|22372   |
|JFK|21832   |
|BWI|21262   |
+---+--------+



### In-degree
Cell này xếp hạng sân bay theo số chuyến bay đi vào; số lớn nghĩa là sân bay nhận nhiều chuyến hơn.

In [154]:
# outDegrees
#  The number of degrees - the number of outgoing connections - for various airports within this sample dataset
tripGraph.outDegrees.sort(desc("outDegree")).limit(20).show(20, truncate=False)

+---+---------+
|id |outDegree|
+---+---------+
|ATL|90141    |
|DFW|68199    |
|ORD|63751    |
|LAX|53669    |
|DEN|52961    |
|IAH|43173    |
|PHX|40053    |
|SFO|38994    |
|LAS|33107    |
|CLT|28059    |
|MCO|27229    |
|EWR|27206    |
|SLC|25611    |
|LGA|25458    |
|BOS|24963    |
|MSP|23863    |
|DTW|23408    |
|SEA|22910    |
|JFK|21829    |
|BWI|21264    |
+---+---------+



### Out-degree
Cell này xếp hạng sân bay theo số chuyến bay đi ra; số lớn nghĩa là sân bay có nhiều tuyến xuất phát hơn.

## City / Flight Relationships through Motif Finding
To more easily understand the complex relationship of city airports and their flights with each other, we can use motifs to find patterns of airports (i.e. vertices) connected by flights (i.e. edges). The result is a DataFrame in which the column names are given by the motif keys.

### Tìm quan hệ thành phố/chuyến bay bằng motif
Phần này tìm các chuỗi bay có dạng A -> SFO -> C để phân tích mối liên hệ giữa các chuyến bay nối chuyến.

#### What delays might we blame on SFO

### Xem motif liên quan đến SFO
Cell này lọc các motif có SFO làm điểm trung chuyển; output là các cấu trúc dữ liệu mô tả các chuyến bay liên tiếp.

In [177]:
# Using tripGraphPrime to more easily display 
#   - The associated edge (ab, bc) relationships 
#   - With the different the city / airports (a, b, c) where SFO is the connecting city (b)
#   - Ensuring that flight ab (i.e. the flight to SFO) occured before flight bc (i.e. flight leaving SFO)
#   - Note, TripID was generated based on time in the format of MMDDHHMM converted to int
#       - Therefore bc.tripid < ab.tripid + 10000 means the second flight (bc) occured within approx a day of the first flight (ab)
# Note: In reality, we would need to be more careful to link trips ab and bc.
motifs = tripGraphPrime.find("(a)-[ab]->(b); (b)-[bc]->(c)")\
  .filter("(b.id = 'SFO') and (ab.delay > 500 or bc.delay > 500) and bc.tripid > ab.tripid and bc.tripid < ab.tripid + 10000")
motifs.show(10, truncate=False)

+---------------------------+------------------------+-----------------------------+------------------------+------------------------+
|a                          |ab                      |b                            |bc                      |c                       |
+---------------------------+------------------------+-----------------------------+------------------------+------------------------+
|{ABQ, Albuquerque, NM, USA}|{1020600, 0, ABQ, SFO}  |{SFO, San Francisco, CA, USA}|{1021507, 536, SFO, JFK}|{JFK, New York, NY, USA}|
|{ABQ, Albuquerque, NM, USA}|{1210815, -12, ABQ, SFO}|{SFO, San Francisco, CA, USA}|{1211508, 593, SFO, JFK}|{JFK, New York, NY, USA}|
|{ACV, Eureka, CA, USA}     |{1011635, -15, ACV, SFO}|{SFO, San Francisco, CA, USA}|{1021507, 536, SFO, JFK}|{JFK, New York, NY, USA}|
|{ACV, Eureka, CA, USA}     |{1012016, -4, ACV, SFO} |{SFO, San Francisco, CA, USA}|{1021507, 536, SFO, JFK}|{JFK, New York, NY, USA}|
|{ACV, Eureka, CA, USA}     |{1020531, -2, ACV, SFO} |{

### Tìm các chuỗi bay có delay lớn qua SFO
Cell này tìm các cặp chuyến bay nối tiếp qua SFO, nơi một trong hai chuyến có delay lớn hơn 500 phút.

## Determining Airport Ranking using PageRank
There are a large number of flights and connections through these various airports included in this Departure Delay Dataset.  Using the `pageRank` algorithm, Spark iteratively traverses the graph and determines a rough estimate of how important the airport is.

### Xếp hạng sân bay bằng PageRank
Cell này tính mức độ quan trọng tương đối của sân bay trong mạng bay; số pagerank càng lớn thì sân bay càng trung tâm.

In [178]:
# Determining Airport ranking of importance using `pageRank`
ranks = tripGraph.pageRank(resetProbability=0.15, maxIter=5)

### Chạy PageRank trên đồ thị
Cell này tính lại PageRank cho toàn bộ sân bay; kết quả số là điểm xếp hạng tương đối của từng sân bay.

## Most popular flights (single city hops)
Using the `tripGraph`, we can quickly determine what are the most popular single city hop flights

### Tìm các chuyến bay phổ biến nhất
Cell này đếm số lượt bay theo từng cặp sân bay nguồn-đích; số trips là số chuyến bay quan sát được.

In [179]:
# Determine the most popular flights (single city hops)
import pyspark.sql.functions as func
topTrips = tripGraph \
  .edges \
  .groupBy("src", "dst") \
  .agg(func.count("delay").alias("trips")) 

### Tổng hợp số chuyến theo tuyến bay
Cell này nhóm theo `src` và `dst`, rồi đếm số chuyến trong mỗi tuyến.

In [180]:
# Show the top 20 most popular flights (single city hops)
topTrips.orderBy(topTrips.trips.desc()).limit(10).show(10, truncate=False)

+---+---+-----+
|src|dst|trips|
+---+---+-----+
|SFO|LAX|3232 |
|LAX|SFO|3198 |
|LAS|LAX|3016 |
|LAX|LAS|2964 |
|JFK|LAX|2720 |
|LAX|JFK|2719 |
|ATL|LGA|2501 |
|LGA|ATL|2500 |
|LAX|PHX|2394 |
|PHX|LAX|2387 |
+---+---+-----+



### Top 10 tuyến bay phổ biến nhất
Bảng in ra sắp xếp theo `trips`; số lớn hơn nghĩa là tuyến bay đó xuất hiện nhiều hơn trong dữ liệu.

## Top Transfer Cities
Many airports are used as transfer points instead of the final Destination.  An easy way to calculate this is by calculating the ratio of inDegree (the number of flights to the airport) / outDegree (the number of flights leaving the airport).  Values close to 1 may indicate many transfers, whereas values < 1 indicate many outgoing flights and > 1 indicate many incoming flights.  Note, this is a simple calculation that does not take into account of timing or scheduling of flights, just the overall aggregate number within the dataset.

### Tính tỷ lệ sân bay trung chuyển
Cell này so sánh in-degree và out-degree để xem sân bay nào gần cân bằng giữa chuyến vào và chuyến ra.

In [181]:
# Calculate the inDeg (flights into the airport) and outDeg (flights leaving the airport)
inDeg = tripGraph.inDegrees
outDeg = tripGraph.outDegrees

# Calculate the degreeRatio (inDeg/outDeg)
degreeRatio = inDeg.join(outDeg, inDeg.id == outDeg.id) \
  .drop(outDeg.id) \
  .selectExpr("id", "double(inDegree)/double(outDegree) as degreeRatio") \
  .cache()

# Join back to the `airports` DataFrame (instead of registering temp table as above)
nonTransferAirports = degreeRatio.join(airports, degreeRatio.id == airports.IATA) \
  .selectExpr("id", "city", "degreeRatio") \
  .filter("degreeRatio < .9 or degreeRatio > 1.1")

# List out the city airports which have abnormal degree ratios.
nonTransferAirports.show(20, truncate=False)

+---+-----------+-------------------+
|id |city       |degreeRatio        |
+---+-----------+-------------------+
|GFK|Grand Forks|1.3333333333333333 |
|FAI|Fairbanks  |1.1232686980609419 |
|OME|Nome       |0.5084745762711864 |
|BRW|Barrow     |0.28651685393258425|
+---+-----------+-------------------+



### Sân bay không phải trung chuyển điển hình
Cell này lọc các sân bay có tỷ lệ in/out lệch xa 1; con số `degreeRatio` càng xa 1 thì sân bay càng ít giống nút trung chuyển.

In [182]:
# Join back to the `airports` DataFrame (instead of registering temp table as above)
transferAirports = degreeRatio.join(airports, degreeRatio.id == airports.IATA) \
  .selectExpr("id", "city", "degreeRatio") \
  .filter("degreeRatio between 0.9 and 1.1")
  
# List out the top 10 transfer city airports
transferAirports.orderBy("degreeRatio").limit(10).show(10, truncate=False)

+---+--------------+------------------+
|id |city          |degreeRatio       |
+---+--------------+------------------+
|MSP|Minneapolis   |0.9375183338222353|
|DEN|Denver        |0.958025717037065 |
|DFW|Dallas        |0.964339653074092 |
|ORD|Chicago       |0.9671063983310065|
|SLC|Salt Lake City|0.9827417906368358|
|IAH|Houston       |0.9846895050147083|
|PHX|Phoenix       |0.9891643572266746|
|OGG|Kahului, Maui |0.9898718478710211|
|HNL|Honolulu, Oahu|0.990535889872173 |
|SFO|San Francisco |0.9909473252295224|
+---+--------------+------------------+



### Sân bay có xu hướng trung chuyển
Cell này lấy các sân bay có `degreeRatio` gần 1; số đó cho biết sân bay cân bằng giữa chuyến vào và chuyến ra.

## Breadth First Search 
Breadth-first search (BFS) is designed to traverse the graph to quickly find the desired vertices (i.e. airports) and edges (i.e flights).  Let's try to find the shortest number of connections between cities based on the dataset.  Note, these examples do not take into account of time or distance, just hops between cities.

### Tìm đường đi ngắn nhất bằng BFS
BFS tìm số chặng ít nhất giữa hai sân bay; output cho biết đường bay trực tiếp hay phải nối chuyến.

### Tìm đường đi ngắn nhất bằng BFS
Breadth-first search dùng để tìm số bước nối ít nhất giữa hai sân bay.

In [183]:
# Example 1: Direct Seattle to San Francisco 
filteredPaths = tripGraph.bfs(
  fromExpr = "id = 'SEA'",
  toExpr = "id = 'SFO'",
  maxPathLength = 1)
filteredPaths.show(10, truncate=False)

+-----------------------+------------------------------------------+-----------------------------+
|v0                     |e0                                        |v1                           |
+-----------------------+------------------------------------------+-----------------------------+
|{SEA, Seattle, WA, USA}|{1010710, 31, SEA, SFO, San Francisco, CA}|{SFO, San Francisco, CA, USA}|
|{SEA, Seattle, WA, USA}|{1012125, -4, SEA, SFO, San Francisco, CA}|{SFO, San Francisco, CA, USA}|
|{SEA, Seattle, WA, USA}|{1011840, -5, SEA, SFO, San Francisco, CA}|{SFO, San Francisco, CA, USA}|
|{SEA, Seattle, WA, USA}|{1010610, -4, SEA, SFO, San Francisco, CA}|{SFO, San Francisco, CA, USA}|
|{SEA, Seattle, WA, USA}|{1011230, -2, SEA, SFO, San Francisco, CA}|{SFO, San Francisco, CA, USA}|
|{SEA, Seattle, WA, USA}|{1010955, -6, SEA, SFO, San Francisco, CA}|{SFO, San Francisco, CA, USA}|
|{SEA, Seattle, WA, USA}|{1011100, 2, SEA, SFO, San Francisco, CA} |{SFO, San Francisco, CA, USA}|
|{SEA, Sea

### Tìm đường bay trực tiếp từ SEA đến SFO
Nếu có kết quả, mỗi dòng là một đường đi trực tiếp; không có số đếm ở đây, chỉ là cấu trúc của path.

As you can see, there are a number of direct flights between Seattle and San Francisco.

### Tìm đường bay từ SFO đến BUF qua tối đa 2 chặng
Cell này tìm đường bay có thể phải nối chuyến; nếu có, output mô tả chuỗi sân bay và cạnh bay.

In [184]:
# Example 2: Direct San Francisco and Buffalo
filteredPaths = tripGraph.bfs(
  fromExpr = "id = 'SFO'",
  toExpr = "id = 'BUF'",
  maxPathLength = 2)
filteredPaths.show(10, truncate=False)

+-----------------------------+----------------------------------+----------------------+------------------------------------+-----------------------------------+
|v0                           |e0                                |v1                    |e1                                  |v2                                 |
+-----------------------------+----------------------------------+----------------------+------------------------------------+-----------------------------------+
|{SFO, San Francisco, CA, USA}|{3312300, 9, SFO, BOS, Boston, MA}|{BOS, Boston, MA, USA}|{1010635, -6, BOS, BUF, Buffalo, NY}|{INL, International Falls, MN, USA}|
|{SFO, San Francisco, CA, USA}|{3312300, 9, SFO, BOS, Boston, MA}|{BOS, Boston, MA, USA}|{1010635, -6, BOS, BUF, Buffalo, NY}|{MSY, New Orleans, LA, USA}        |
|{SFO, San Francisco, CA, USA}|{3312300, 9, SFO, BOS, Boston, MA}|{BOS, Boston, MA, USA}|{1010635, -6, BOS, BUF, Buffalo, NY}|{GEG, Spokane, WA, USA}            |
|{SFO, San Francisco, 

### BFS từ SFO đến BUF
Cell này tìm các path SFO -> BUF trong tối đa 2 chặng; không phải số liệu tổng hợp mà là các đường đi cụ thể.

In [185]:
# Display most popular layover cities by descending count
filteredPaths.groupBy("v1.id", "v1.City").count().orderBy(desc("count")).limit(10).show(10, truncate=False)

+---+---------------+---------+
|id |City           |count    |
+---+---------------+---------+
|JFK|New York       |344210112|
|ORD|Chicago        |303630957|
|ATL|Atlanta        |79621857 |
|LAS|Las Vegas      |76750389 |
|BOS|Boston         |66562704 |
|CLT|Charlotte      |40020876 |
|PHX|Phoenix        |29177820 |
|FLL|Fort Lauderdale|26872443 |
|EWR|Newark         |26608230 |
|MCO|Orlando        |24723585 |
+---+---------------+---------+



### Đếm các điểm trung chuyển phổ biến
Bảng đếm cho biết sân bay trung gian nào xuất hiện nhiều nhất trên các path BFS đã tìm được.

In [186]:
# Example 2: Direct San Francisco and Buffalo
filteredPaths = tripGraph.bfs(
  fromExpr = "id = 'SFO'",
  toExpr = "id = 'BUF'",
  maxPathLength = 1)
filteredPaths.show(10, truncate=False)

+---+---+---+
|v0 |e0 |v1 |
+---+---+---+
+---+---+---+



### Kiểm tra không có chuyến bay trực tiếp SFO -> BUF
Cell này thử BFS với 1 chặng; nếu không có dòng nào thì nghĩa là không có chuyến bay thẳng.

But there are no direct flights between San Francisco and Buffalo.

### Tìm đường bay SFO -> BUF với tối đa 2 chặng
Cell này cho phép nối chuyến; output mô tả các path có thể đi qua một sân bay trung gian.

In [187]:
# Example 2a: Flying from San Francisco to Buffalo
filteredPaths = tripGraph.bfs(
  fromExpr = "id = 'SFO'",
  toExpr = "id = 'BUF'",
  maxPathLength = 2)
filteredPaths.show(10, truncate=False)

+-----------------------------+----------------------------------+----------------------+------------------------------------+-----------------------------------+
|v0                           |e0                                |v1                    |e1                                  |v2                                 |
+-----------------------------+----------------------------------+----------------------+------------------------------------+-----------------------------------+
|{SFO, San Francisco, CA, USA}|{3312300, 9, SFO, BOS, Boston, MA}|{BOS, Boston, MA, USA}|{1010635, -6, BOS, BUF, Buffalo, NY}|{INL, International Falls, MN, USA}|
|{SFO, San Francisco, CA, USA}|{3312300, 9, SFO, BOS, Boston, MA}|{BOS, Boston, MA, USA}|{1010635, -6, BOS, BUF, Buffalo, NY}|{MSY, New Orleans, LA, USA}        |
|{SFO, San Francisco, CA, USA}|{3312300, 9, SFO, BOS, Boston, MA}|{BOS, Boston, MA, USA}|{1010635, -6, BOS, BUF, Buffalo, NY}|{GEG, Spokane, WA, USA}            |
|{SFO, San Francisco, 

But there are flights from San Francisco to Buffalo with Minneapolis as the transfer point.  But what are the most popular layovers between `SFO` and `BUF`?

In [131]:
# Display most popular layover cities by descending count
filteredPaths.groupBy("v1.id", "v1.City").count().orderBy(desc("count")).limit(10).show(10, truncate=False)

+---+---------------+---------+
|id |City           |count    |
+---+---------------+---------+
|JFK|New York       |344210112|
|ORD|Chicago        |303630957|
|ATL|Atlanta        |79621857 |
|LAS|Las Vegas      |76750389 |
|BOS|Boston         |66562704 |
|CLT|Charlotte      |40020876 |
|PHX|Phoenix        |29177820 |
|FLL|Fort Lauderdale|26872443 |
|EWR|Newark         |26608230 |
|MCO|Orlando        |24723585 |
+---+---------------+---------+



### Xem trung chuyển phổ biến nhất giữa SFO và BUF
Bảng đếm cho biết sân bay trung gian nào xuất hiện nhiều nhất trong các đường bay SFO -> BUF.

## Loading the D3 Visualization
Using the airports D3 visualization to visualize airports and flight paths

### Ghi chú về file map hãng bay
Notebook gốc dùng một file map ngoài để ghép tripid với airline; bản local này thay bằng cách tạo nhãn hãng bay đơn giản từ tuyến bay.

In [188]:
# Prep dataset
# Limiting to only Las Vegas (LAS) and Seattle (SEA) predictions so it can run locally
flightML = spark.sql("""
select cast(distance as double) as distance,
       src as origin,
       state_src as origin_state,
       dst as destination,
       state_dst as destination_state,
       concat(cast(tripid as string), src, dst, cast((delay + 2000) as string)) as trip_identifier,
       concat(src, '_', dst) as airline,
       case when delay <= 0 then 'on-time' else 'delayed' end as flight_status
from departureDelays_geo
where src in ('LAS', 'SEA')
""")
flightML = flightML.dropna().dropDuplicates()
flightML.createOrReplaceTempView("flightML")

### Tạo tập dữ liệu học máy
Cell này lọc các chuyến đi từ LAS và SEA, tạo nhãn `flight_status` và cột `trip_identifier` để dùng cho mô hình.

In [189]:
# Keep the prepared dataset directly; the airline feature is derived locally above.
dataset = flightML.dropDuplicates()
cols = dataset.columns

### Tạo dataset cuối cùng cho ML
Cell này giữ lại tập dữ liệu đã làm sạch để huấn luyện; kết quả là các cột đặc trưng và nhãn cần thiết cho mô hình.

In [190]:
dataset.printSchema()

root
 |-- distance: double (nullable = true)
 |-- origin: string (nullable = true)
 |-- origin_state: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- destination_state: string (nullable = true)
 |-- trip_identifier: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- flight_status: string (nullable = false)



### Kiểm tra schema của dataset ML
Cell này in schema để xác nhận kiểu dữ liệu của từng cột trước khi đưa vào pipeline.

In [191]:
dataset.count()

55363

### Đếm số mẫu huấn luyện
Con số trả về là số dòng hiện có trong dataset ML trước khi chia train/test.

### Building ML Pipeline
Before we can run our various models against this data, we will first need to vectorize our data via One-Hot Encorder (for category data), String Indexer (create an index based on our labelled values), and Vector Assembler.

### Xây dựng pipeline ML
Phần này biến các cột phân loại thành vector one-hot, tạo nhãn số và ghép tất cả đặc trưng vào một cột `features`.

In [192]:
# One-Hot Encoding
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler

categoricalColumns = ["origin", "origin_state", "destination", "destination_state", "trip_identifier", "airline"]
#categoricalColumns = ["origin", "origin_state", "destination", "destination_state", "trip_identifier"]
stages = [] # stages in our Pipeline
for categoricalCol in categoricalColumns:
  # Category Indexing with StringIndexer
  stringIndexer = StringIndexer(inputCol=categoricalCol, outputCol=categoricalCol+"Index")
  
  # Use OneHotEncoder to convert categorical variables into binary SparseVectors
  encoder = OneHotEncoder(inputCol=categoricalCol+"Index", outputCol=categoricalCol+"classVec")
  
  # Add stages.  These are not run here, but will run all at once later on.
  stages += [stringIndexer, encoder]

# Convert label into label indices using the StringIndexer
label_stringIdx = StringIndexer(inputCol = "flight_status", outputCol = "label")
stages += [label_stringIdx]

# Transform all features into a vector using VectorAssembler
numericCols = ["distance"]
assemblerInputs = list(map(lambda c: c + "classVec", categoricalColumns)) + numericCols
assembler = VectorAssembler(inputCols=assemblerInputs, outputCol="features")
stages += [assembler]

### Cấu hình pipeline đặc trưng
Cell này khai báo các bước StringIndexer, OneHotEncoder và VectorAssembler để biến dữ liệu dạng bảng thành vector cho mô hình.

In [193]:
# Create a Pipeline.
dataset = flightML.dropDuplicates()
cols = dataset.columns
pipeline = Pipeline(stages=stages)
# Run the feature transformations.
#  - fit() computes feature statistics as needed.
#  - transform() actually transforms the features.
pipelineModel = pipeline.fit(dataset)
dataset = pipelineModel.transform(dataset)

# Keep relevant columns
selectedcols = ["label", "features"] + cols
dataset = dataset.select(selectedcols)
dataset.show(10, truncate=False)

+-----+----------------------------------------------------------------------+--------+------+------------+-----------+-----------------+-----------------+-------+-------------+
|label|features                                                              |distance|origin|origin_state|destination|destination_state|trip_identifier  |airline|flight_status|
+-----+----------------------------------------------------------------------+--------+------+------------+-----------+-----------------+-----------------+-------+-------------+
|0.0  |(55611,[0,1,8,88,2898,55505,55610],[1.0,1.0,1.0,1.0,1.0,1.0,1316.0])  |1316.0  |LAS   |NV          |ORD        |IL               |1051220LASORD2000|LAS_ORD|on-time      |
|0.0  |(55611,[0,1,35,98,4560,55532,55610],[1.0,1.0,1.0,1.0,1.0,1.0,1889.0]) |1889.0  |LAS   |NV          |MIA        |FL               |1080720LASMIA1998|LAS_MIA|on-time      |
|1.0  |(55611,[0,1,35,98,15410,55532,55610],[1.0,1.0,1.0,1.0,1.0,1.0,1889.0])|1889.0  |LAS   |NV          |MIA

### Chạy pipeline lên dữ liệu
Cell này fit pipeline rồi transform dữ liệu; bảng kết quả sẽ có `label` và `features` để huấn luyện.

### Randomly split data into training and test datasets
* Set the seed for reproducibility

### Chia train/test
Cell này chia dataset thành tập huấn luyện và tập kiểm tra; hai số in ra là số dòng của mỗi tập.

In [194]:
(trainingData, testData) = dataset.randomSplit([0.7, 0.3], seed = 100)
print(trainingData.count())
print(testData.count())

38759
16604


### Chia dữ liệu thành train và test
Hai số in ra là số dòng trong tập huấn luyện và tập kiểm tra sau khi chia ngẫu nhiên.

## Logistic Regression
Let's try using logistic regression to see if we can accurately predict if a flight will be delayed.
* First, we will train the data using Logistic Regression
* Next we will run that model against the testData

### Huấn luyện Logistic Regression
Cell này fit mô hình logistic regression để dự đoán chuyến bay có bị trễ hay không.

In [195]:
from pyspark.ml.classification import LogisticRegression

# Create initial LogisticRegression model
lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=10)

# Train model with Training Data
lrModel = lr.fit(trainingData)

### Tạo mô hình Logistic Regression
Cell này khởi tạo mô hình logistic regression với các đặc trưng đã vector hóa.

In [196]:
# Make predictions on test data using the transform() method.
# LogisticRegression.transform() will only use the 'features' column.
predictions = lrModel.transform(testData)

### Chạy mô hình trên tập kiểm tra
Cell này dùng mô hình đã học để tạo dự đoán trên `testData`.

### View LR Model's predictions
* Recall, label is the actual test value, prediction is the predicted value
 * where 0 - on-time, 1 - delayed

### Xem dự đoán của mô hình
`label` là nhãn thật, `prediction` là nhãn dự đoán, `probability` là xác suất cho từng lớp.

In [197]:
selected = predictions.select("label", "prediction", "probability", "flight_status", "destination", "destination_state").where("destination = 'SEA'")
selected.show(10, truncate=False)

+-----+----------+---------------------------------------+-------------+-----------+-----------------+
|label|prediction|probability                            |flight_status|destination|destination_state|
+-----+----------+---------------------------------------+-------------+-----------+-----------------+
|0.0  |0.0       |[0.846931461043865,0.15306853895613504]|on-time      |SEA        |WA               |
|0.0  |0.0       |[0.846931461043865,0.15306853895613504]|on-time      |SEA        |WA               |
|0.0  |0.0       |[0.846931461043865,0.15306853895613504]|on-time      |SEA        |WA               |
|0.0  |0.0       |[0.846931461043865,0.15306853895613504]|on-time      |SEA        |WA               |
|0.0  |0.0       |[0.846931461043865,0.15306853895613504]|on-time      |SEA        |WA               |
|0.0  |0.0       |[0.846931461043865,0.15306853895613504]|on-time      |SEA        |WA               |
|0.0  |0.0       |[0.846931461043865,0.15306853895613504]|on-time      |S

### Xem dự đoán cho điểm đến SEA
Cell này lọc các dòng có điểm đến SEA để xem mô hình dự đoán ra sao cho nhóm đó.

#### Evaluate our model
Let's use the `BinaryClassificationEvaluator` to determine the precision of our model.

### Đánh giá mô hình
Cell này dùng BinaryClassificationEvaluator để tính điểm đánh giá trên tập dự đoán.

In [198]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Evaluate model
evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction")
evaluator.evaluate(predictions)

0.6481554175744794

### Tính điểm đánh giá
Giá trị số trả về là điểm chất lượng của mô hình phân loại nhị phân trên tập dự đoán.